In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_google_genai import ChatGoogleGenerativeAI

In [15]:
documents = [

    Document(
        page_content="""
        Breakfast Guide

        Oatmeal is a healthy breakfast option.
        It is rich in fiber and can keep you full for a long time.
        You can add bananas, apples, nuts, or honey to oatmeal.
        Oatmeal usually takes about five minutes to prepare.

        Oatmeal can also be eaten as an evening snack.
        Many people prefer eating it during winter because it is warm.
        """,
        metadata={"topic": "oatmeal"}
    ),

    Document(
        page_content="""
        Breakfast Ideas

        Eggs are a popular breakfast food.
        They are a good source of protein.
        You can prepare eggs as an omelette, scrambled eggs, or boiled eggs.
        Eggs can be combined with vegetables and bread.

        Eggs are also commonly used in sandwiches.
        They should be stored in a refrigerator.
        """,
        metadata={"topic": "eggs"}
    ),

    Document(
        page_content="""
        Fruit Breakfast

        Bananas are convenient breakfast foods.
        They are easy to carry and require no preparation.
        Bananas provide carbohydrates and potassium.
        They can be eaten alone or added to cereal and oatmeal.

        Bananas can also be used in smoothies.
        Ripe bananas are sweeter than green bananas.
        """,
        metadata={"topic": "fruit"}
    )
]

In [16]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS Vector store from documents
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7914.65it/s]


In [17]:
base_retriever = vectorstore.as_retriever(search_kwargs={'k':5})

In [23]:
llm = ChatGoogleGenerativeAI(
    model="models/gemini-3.6-flash",
    temperature=0
)

compressor = LLMChainExtractor.from_llm(llm)

In [24]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = compressor
)

In [25]:
query = "Which breakfast foods are good sources of protein?"

In [26]:
print("\n========== NORMAL RETRIEVER ==========\n")

normal_results = base_retriever.invoke(query)

for doc in normal_results:

    print("TOPIC:", doc.metadata["topic"])
    print(doc.page_content)
    print("-" * 60)



========== NORMAL RETRIEVER ==========

TOPIC: eggs

        Breakfast Ideas

        Eggs are a popular breakfast food.
        They are a good source of protein.
        You can prepare eggs as an omelette, scrambled eggs, or boiled eggs.
        Eggs can be combined with vegetables and bread.

        Eggs are also commonly used in sandwiches.
        They should be stored in a refrigerator.
        
------------------------------------------------------------
TOPIC: oatmeal

        Breakfast Guide

        Oatmeal is a healthy breakfast option.
        It is rich in fiber and can keep you full for a long time.
        You can add bananas, apples, nuts, or honey to oatmeal.
        Oatmeal usually takes about five minutes to prepare.

        Oatmeal can also be eaten as an evening snack.
        Many people prefer eating it during winter because it is warm.
        
------------------------------------------------------------
TOPIC: fruit

        Fruit Breakfast

        Bananas

In [27]:
print("\n========== COMPRESSED RETRIEVER ==========\n")

compressed_results = compression_retriever.invoke(query)

for doc in compressed_results:

    print("TOPIC:", doc.metadata["topic"])
    print(doc.page_content)
    print("-" * 60)


========== COMPRESSED RETRIEVER ==========



c:\langchainModels\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\langchainModels\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\langchainModels\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


TOPIC: eggs
Eggs are a popular breakfast food.
They are a good source of protein.
------------------------------------------------------------
